In [1]:
import pandas as pd
import numpy as np

scoring = pd.read_csv("../data_processed/scoring_with_arrangement.csv", na_values=["NA"])
par_pools = pd.read_csv("../data_raw/par_pools.csv", na_values=["NA"])

print(par_pools.shape)
print(scoring.shape)


(5, 3)
(36000, 10)


In [2]:
def assign_pd_pool(pd_value, par_pools):
    for r, row in par_pools.iterrows():
        if pd_value >= row["START"] and pd_value < row["END"]:
            return row["LABEL"]
    return np.nan

test_pd_value = 0.0009
print("par_pools LABEL", assign_pd_pool(test_pd_value, par_pools))

par_pools LABEL 2.0


In [3]:
expected_pools = []

for pd_value in scoring["PD"]:
    if pd.isna(pd_value):
        expected_pools.append(np.nan)
    else:
        expected_pools.append(assign_pd_pool(pd_value, par_pools))
        
scoring["expected_pd_pool"] = expected_pools
scoring[["PD", "PD_POOL", "expected_pd_pool"]].head(10)

,PD,PD_POOL,expected_pd_pool
0,9.925260e-01,5.0,5.0
1,2.656939e-05,1.0,1.0
2,3.514735e-06,1.0,1.0
3,5.868028e-06,1.0,1.0
4,8.378070e-07,1.0,1.0
5,2.703949e-07,1.0,1.0
6,1.157728e-06,1.0,1.0
7,3.515353e-07,1.0,1.0
8,5.125351e-07,1.0,1.0
9,5.590088e-01,5.0,5.0


In [4]:
(scoring["PD_POOL"] == scoring["expected_pd_pool"]).value_counts()

True     35105
False      895
Name: count, dtype: int64

In [5]:
mismatch = scoring[scoring['PD_POOL'] != scoring['expected_pd_pool']]

mismatch[['YEAR', 'AR_ID', 'PD', 'PD_POOL', 'expected_pd_pool']].head(20)

,YEAR,AR_ID,PD,PD_POOL,expected_pd_pool
12,2022,14188754,NaN,NaN,NaN
21,2022,8302079,NaN,NaN,NaN
37,2022,99211895,NaN,NaN,NaN
51,2022,41935251,NaN,NaN,NaN
63,2022,48588518,NaN,NaN,NaN
65,2022,63967549,NaN,NaN,NaN
100,2022,22332720,NaN,NaN,NaN
221,2022,53452186,NaN,NaN,NaN
243,2022,69925664,NaN,NaN,NaN
302,2022,97429589,NaN,NaN,NaN


In [6]:
compare_notna = scoring[scoring['PD_POOL'].notna() & scoring['expected_pd_pool'].notna()]

mismatch_notna = compare_notna[compare_notna['PD_POOL'] != compare_notna['expected_pd_pool']]

mismatch_notna.shape

(0, 11)

In [7]:
pd_missing = scoring[scoring['PD'].isna()]

pd_missing.shape

(895, 11)

In [8]:
pd_missing.groupby('YEAR')['DFLT_FLAG'].mean()

YEAR
2022    0.115385
2023    0.115512
2024    0.336601
Name: DFLT_FLAG, dtype: float64

In [11]:
pd_available = scoring['PD'].notna()

scoring.groupby(['YEAR', scoring['PD'].notna()])['DFLT_FLAG'].mean()

YEAR  PD   
2022  False    0.115385
      True     0.127625
2023  False    0.115512
      True     0.124647
2024  False    0.336601
      True     0.345989
Name: DFLT_FLAG, dtype: float64

In [12]:
pd_available = scoring[scoring['PD'].notna()].copy()
pd_missing = scoring[scoring["PD"].isna()].copy()

print("PD available:", pd_available.shape)
print("PD missing:", pd_missing.shape)

PD available: (35105, 11)
PD missing: (895, 11)


In [13]:
n_by_pool = pd_available.groupby("PD_POOL")["AR_ID"].count()
default_by_pool = pd_available.groupby("PD_POOL")["DFLT_FLAG"].mean()
pd_by_pool = pd_available.groupby("PD_POOL")["PD"].mean()

pool_overall = pd.DataFrame({
    "n": n_by_pool,
    "default_rate": default_by_pool,
    "avg_pd": pd_by_pool
})

pool_overall = pool_overall.reset_index()
pool_overall

,PD_POOL,n,default_rate,avg_pd
0,1.0,16784,0.031220,0.000168
1,2.0,5172,0.074633,0.001581
2,3.0,4724,0.074090,0.005513
3,4.0,2940,0.103741,0.016931
4,5.0,5485,0.990702,0.915417


In [14]:
n_by_year_pool = pd_available.groupby(["YEAR", "PD_POOL"])["AR_ID"].count()
default_by_year_pool = pd_available.groupby(["YEAR", "PD_POOL"])["DFLT_FLAG"].mean()
pd_by_year_pool = pd_available.groupby(["YEAR", "PD_POOL"])["PD"].mean()

pool_year = pd.DataFrame({
    "n": n_by_year_pool,
    "default_rate": default_by_year_pool,
    "avg_pd": pd_by_year_pool
})

pool_year = pool_year.reset_index()
pool_year

,YEAR,PD_POOL,n,default_rate,avg_pd
0,2022,1.0,10262,0.005165,0.000004
1,2022,2.0,7,0.428571,0.001852
2,2022,3.0,10,1.000000,0.005862
3,2022,4.0,60,0.966667,0.028885
4,2022,5.0,1375,0.997091,0.702280
5,2023,1.0,3386,0.002658,0.000425
6,2023,2.0,2816,0.003551,0.001598
7,2023,3.0,2535,0.008679,0.005466
8,2023,4.0,1539,0.017544,0.016638
9,2023,5.0,1421,0.978184,0.979340
